# Nell Finder

## Before you start

Add your images and matching polygon labels to the input folders.
Each label file should have the same name as its image, only with `.txt` at the end.
If you still need to create labels, [Label Studio](https://labelstud.io/) is a good option.


```
nell-finder-tr/
├── data/
│   ├── input/
│   │   ├── images/   ← add your images here
│   │   └── labels/   ← add your labels here
│   └── tmp/          ← generated data.yaml, train.txt, val.txt
├── dist/             ← exported model is copied here
├── runs/             ← Ultralytics training + prediction runs
└── src/
    ├── classes.txt   ← class names, one per line
    ├── notes.json
    └── nell_finder_local.ipynb   ← this notebook
```

Run the notebook from top to bottom. Most of the time you do not need to change anything.
For a first test, the only thing you usually change is the sample image near the end.

## Environment

This checks that the required packages load and chooses the best available device automatically.

In [ ]:
# IMPORTS
import io
from contextlib import redirect_stderr, redirect_stdout

# LOAD THE REQUIRED PACKAGES QUIETLY
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    import torch
    import ultralytics
    ultralytics.checks(verbose=False)

# CHOOSE THE BEST AVAILABLE DEVICE
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# PRINT THE DEVICE USED FOR TRAINING AND PREDICTION
print("DEVICE:", DEVICE)

## Find the project

This finds the project folder automatically and makes sure the required inputs are there before anything else runs.

In [ ]:
# IMPORTS
from pathlib import Path

# FUNCTION: FIND THE PROJECT ROOT
def find_root(start: Path) -> Path:

    # GET ALL DIRS
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "src").is_dir():
            return p

    # RAISE ERROR IF FILES NOT FOUND
    raise FileNotFoundError(
        "Could not find the project root (a folder with both data/ and src/). "
        f"Started looking from: {start}"
    )

# LOCATE THE PROJECT ROOT
PROJECT_ROOT = find_root(Path.cwd())

# DEFINE THE PROJECT PATHS
IMAGES_DIR  = PROJECT_ROOT / "data" / "input" / "images"
LABELS_DIR  = PROJECT_ROOT / "data" / "input" / "labels"
CLASSES_TXT = PROJECT_ROOT / "src"  / "classes.txt"
TMP_DIR     = PROJECT_ROOT / "data" / "tmp"
RUNS_DIR    = PROJECT_ROOT / "runs"
DIST_DIR    = PROJECT_ROOT / "dist"

# MAKE SURE THE REQUIRED INPUT PATHS EXIST
required_paths = [
    ("IMAGE DIRECTORY", IMAGES_DIR),
    ("LABEL DIRECTORY", LABELS_DIR),
    ("CLASSES FILE", CLASSES_TXT),
]

# GET MISSING PATSH
missing_paths = [f"{label}: {path}" for label, path in required_paths if not path.exists()]

# RAISE ERROR IF PATHS ARE MISSING
if missing_paths:
    raise FileNotFoundError("Missing required project inputs:\n" + "\n".join(missing_paths))

# PRINT THE RESOLVED PROJECT ROOT
print("PROJECT ROOT:", PROJECT_ROOT)

## Quick data check

This shows a short summary of your data so you can catch missing labels before training starts.

In [ ]:
# DEFINE THE ALLOWED IMAGE EXTENSIONS
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

# COLLECT THE INPUT FILES
images = sorted((p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS), key=lambda p: p.name)
labels = sorted(LABELS_DIR.glob("*.txt"))
class_names = [ln.strip() for ln in CLASSES_TXT.read_text().splitlines() if ln.strip()]

# MAKE SURE THE CLASS LIST IS NOT EMPTY
if not class_names:
    raise ValueError(f"classes.txt is empty: {CLASSES_TXT}")

# COUNT IMAGES THAT STILL DO NOT HAVE A MATCHING LABEL FILE
label_stems = {p.stem for p in labels}
missing_label_count = sum(1 for image_path in images if image_path.stem not in label_stems)

# PRINT A SHORT INPUT SUMMARY
print(
    f"IMAGES: {len(images)} | LABELS: {len(labels)} | CLASSES: {len(class_names)}"
    f" | IMAGES WITHOUT LABEL: {missing_label_count}"
)


## Prepare the dataset

This creates a reproducible train/validation split and writes the small helper files needed for training.
Your original images and labels are left untouched. Files without a matching partner are skipped automatically.


In [ ]:
# IMPORTS
import random

# CREATE THE TEMP DIRECTORY FOR GENERATED FILES
TMP_DIR.mkdir(parents=True, exist_ok=True)

# READ THE CLASS NAMES
class_names = [ln.strip() for ln in CLASSES_TXT.read_text().splitlines() if ln.strip()]
if not class_names:
    raise ValueError(f"classes.txt is empty: {CLASSES_TXT}")

# LIST ALL IMAGES IN A STABLE ORDER
image_files = sorted(
    (p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
    key=lambda p: p.name,
)
if not image_files:
    raise FileNotFoundError(f"No images found in {IMAGES_DIR}")

# PAIR EACH IMAGE WITH ITS MATCHING LABEL FILE
pairs = []
skipped_images = 0
for image_path in image_files:
    label_path = LABELS_DIR / f"{image_path.stem}.txt"
    if not label_path.exists():
        skipped_images += 1
        continue
    pairs.append((image_path, label_path))

# STOP IF NO VALID IMAGE/LABEL PAIRS WERE FOUND
if not pairs:
    raise RuntimeError("No image/label pairs found. Check that image and label stems match.")

# SHUFFLE AFTER SORTING SO THE SPLIT STAYS REPRODUCIBLE
random.seed(42)
random.shuffle(pairs)

# BUILD THE TRAIN / VALIDATION SPLIT
val_count = max(1, int(len(pairs) * 0.2))
val_pairs, train_pairs = pairs[:val_count], pairs[val_count:]

# WRITE THE IMAGE LISTS USED BY ULTRALYTICS
train_txt = TMP_DIR / "train.txt"
val_txt = TMP_DIR / "val.txt"
train_txt.write_text("".join(f"{image_path.resolve()}\n" for image_path, _ in train_pairs), encoding="utf-8")
val_txt.write_text("".join(f"{image_path.resolve()}\n" for image_path, _ in val_pairs), encoding="utf-8")

# WRITE data.yaml WITH AN ABSOLUTE BASE PATH
names_yaml = "\n".join(f"  {i}: {name}" for i, name in enumerate(class_names))
data_yaml = f"path: {TMP_DIR.resolve()}\ntrain: train.txt\nval: val.txt\n\nnames:\n{names_yaml}\n"
(TMP_DIR / "data.yaml").write_text(data_yaml, encoding="utf-8")

# PRINT A SHORT SUMMARY OF THE GENERATED SPLIT
print(
    f"PAIRS: {len(pairs)} | TRAIN: {len(train_pairs)} | VAL: {len(val_pairs)}"
    f" | SKIPPED IMAGES: {skipped_images}"
)


## Train

This starts training and saves each run in its own folder, so earlier runs stay available.
When training finishes, the notebook automatically points to the best weights from that run.

In [ ]:
# IMPORTS
import os
from datetime import datetime
from ultralytics import YOLO

# ALLOW PYTORCH TO FALL BACK TO THE CPU FOR UNSUPPORTED MPS OPERATIONS
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# CREATE A UNIQUE RUN NAME SO TRAINING NEVER OVERWRITES AN EARLIER RUN
RUN_NAME = f"jass-seg-{datetime.now():%Y%m%d-%H%M%S}"

# LOAD THE SEGMENTATION MODEL
model = YOLO("yolo11n-seg.pt")

# TRAIN THE MODEL
results = model.train(
    data=str(TMP_DIR / "data.yaml"),
    epochs=3,
    imgsz=320,
    batch=16,
    device=DEVICE,
    patience=50,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,
)

# DEFINE THE PATH TO THE BEST WEIGHTS
BEST = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

# TO REUSE AN EARLIER RUN, SET BEST MANUALLY TO runs/<that-run>/weights/best.pt
print("BEST WEIGHTS:", BEST)

## Test on one image

Use this after training to try the model on one image.
The result is printed in the notebook and an annotated copy is saved automatically.

In [ ]:
# IMPORTS
from ultralytics import YOLO

# MAKE SURE A TRAINED MODEL IS AVAILABLE
if "BEST" not in globals():
    raise NameError("BEST is not defined. Run the training cell first or set BEST manually.")
if not BEST.exists():
    raise FileNotFoundError(f"Best weights not found: {BEST}")

# LOAD THE TRAINED MODEL
model = YOLO(str(BEST))

# SET THE TEST IMAGE
img_path = IMAGES_DIR / "0Y1A4928.JPG"   # EDIT: USE A REAL FILE FROM data/input/images OR ANOTHER IMAGE
if not img_path.exists():
    raise FileNotFoundError(f"Image not found: {img_path}")

# RUN PREDICTION AND SAVE AN ANNOTATED COPY
res = model.predict(
    str(img_path),
    imgsz=320,
    device=DEVICE,
    save=True,
    project=str(RUNS_DIR),
    name="predict",
    exist_ok=True,
    verbose=False,
)
r = res[0]

# PRINT THE DETECTIONS SORTED BY CONFIDENCE
if r.boxes is None or len(r.boxes) == 0:
    print("NO CARD DETECTED. TRY A CLEARER OR CLOSER IMAGE.")
else:
    confs = r.boxes.conf.cpu().numpy()
    clss = r.boxes.cls.cpu().numpy().astype(int)
    print("DETECTED:")
    for i in confs.argsort()[::-1]:
        print(f"  {r.names[int(clss[i])]:<22} CONF {confs[i]:.2f}")

## Export

This optional step exports the trained model and copies it to the export folder.

In [ ]:
# IMPORTS
import shutil
from pathlib import Path
from ultralytics import YOLO

# MAKE SURE A TRAINED MODEL IS AVAILABLE
if "BEST" not in globals():
    raise NameError("BEST is not defined. Run the training cell first or set BEST manually.")
if not BEST.exists():
    raise FileNotFoundError(f"Best weights not found: {BEST}")

# LOAD THE TRAINED MODEL
model = YOLO(str(BEST))

# EXPORT THE MODEL TO ONNX
exported = Path(model.export(format="onnx", dynamic=True))

# CREATE THE EXPORT DIRECTORY
DIST_DIR.mkdir(parents=True, exist_ok=True)

# BUILD THE FINAL OUTPUT PATH
dest = DIST_DIR / exported.name

# COPY THE EXPORTED FILE INTO dist/
shutil.copy2(exported, dest)

# PRINT THE FINAL EXPORT LOCATION
print("EXPORTED MODEL:", dest)
